In [ ]:
import os
from pathlib import Path

REVIEW_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "REVIEW_CODE_ROOT").is_file()),
    None,
)
if REVIEW_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted review package.")
os.chdir(REVIEW_ROOT)
REVIEW_FREQUENCY = os.environ.get("BFORTPROB_FREQUENCY", "daily")
if REVIEW_FREQUENCY not in {"daily", "weekly"}:
    raise ValueError("BFORTPROB_FREQUENCY must be daily or weekly.")


# Overall CRPS and MASE

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def significance_analysis(csv_path, metric_col, output_prefix):
    # ------------------------------------------------------------
    # 1. Load data
    # ------------------------------------------------------------
    df = pd.read_csv(csv_path)

    print(f"\n=== {output_prefix.upper()} ===")
    print(f"Datasets: {df['dataset'].nunique()}")
    print(f"Models: {df['model'].nunique()}")
    print(df['model'].unique())

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index='dataset',
        columns='model',
        values=metric_col
    )

    # Friedman requires every dataset to contain every model.
    complete = pivot.dropna().rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "GARCH"})

    print(f"Complete datasets used: {len(complete)}")

    if len(complete) != len(pivot):
        print(
            f"WARNING: dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank models within every dataset
    #    1 = lowest metric = best rank
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=True,
        method='average'
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[complete[col].values for col in complete.columns]
    )

    print(
        f"\nFriedman chi-square = {stat:.4f}, "
        f"p = {p_value:.4e}"
    )

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc test
    # ------------------------------------------------------------
    if p_value < 0.05:
        p_matrix = sp.posthoc_nemenyi_friedman(complete)

        print("\nNemenyi pairwise p-values:")
        print(p_matrix)

        # Save pairwise p-values
        p_matrix.to_csv(
            f"{output_prefix}_nemenyi_pvalues.csv"
        )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)
            

        # plt.title(
        #     f"{output_prefix.upper()} Critical Difference Diagram\n"
        #     f"Friedman p = {p_value:.2e}"
        # )

        plt.tight_layout()
        plt.savefig(
            f"{output_prefix}_critical_difference.png",
            dpi=300,
            bbox_inches='tight'
        )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nFriedman test was not significant; "
            "Nemenyi post-hoc comparisons are not interpreted."
        )

    # ------------------------------------------------------------
    # 7. Save ranks
    # ------------------------------------------------------------
    avg_rank_df = (
        avg_ranks
        .rename('average_rank')
        .reset_index()
    )

    avg_rank_df.to_csv(
        f"{output_prefix}_average_ranks.csv",
        index=False
    )

    return complete, ranks, avg_ranks, stat, p_value, p_matrix


# ================================================================
# MASE
# ================================================================
mase_results = significance_analysis(
    csv_path=Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/all_models_mase_by_dataset.csv"),
    metric_col="mean_mase",
    output_prefix="mase"
)


# ================================================================
# CRPS
# ================================================================
crps_results = significance_analysis(
    csv_path=Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/all_models_crps_by_dataset.csv"),
    metric_col="mean_qcrps",
    output_prefix="crps"
)

# Overall RMSE and QSS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


def overall_significance_analysis(
    csv_path,
    metric_col,
    metric_name,
    higher_is_better=False,
    output_prefix=None
):
    df = pd.read_csv(csv_path)

    print("\n" + "=" * 80)
    print(f"OVERALL {metric_name.upper()} SIGNIFICANCE ANALYSIS")
    print("=" * 80)

    print("Datasets:", df["dataset"].nunique())
    print("Models:", df["model"].nunique())
    print("Models:", list(df["model"].unique()))

    # ------------------------------------------------------------
    # 1. Check duplicates
    # ------------------------------------------------------------
    duplicates = df.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nDuplicate dataset/model rows found:")
        print(
            df.loc[
                duplicates,
                ["dataset", "model", metric_col]
            ]
        )
        raise ValueError("Duplicate dataset/model entries detected.")

    # ------------------------------------------------------------
    # 2. Dataset × Model matrix
    # ------------------------------------------------------------
    pivot = df.pivot(
        index="dataset",
        columns="model",
        values=metric_col
    )

    print("\nOriginal matrix shape:", pivot.shape)

    # Friedman needs complete blocks
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "GARCH"})

    print(
        f"Complete datasets used: {len(complete)} / {len(pivot)}"
    )

    if len(complete) != len(pivot):
        print(
            f"Dropped {len(pivot) - len(complete)} datasets "
            "because at least one model was missing."
        )

    # ------------------------------------------------------------
    # 3. Rank within each dataset
    # ------------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=not higher_is_better,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # ------------------------------------------------------------
    # 4. Friedman test
    # ------------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test:")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # ------------------------------------------------------------
    # 5. Nemenyi post-hoc
    # ------------------------------------------------------------
    if p_value < 0.05:

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-values:")
        print(p_matrix)

        if output_prefix is not None:
            p_matrix.to_csv(
                f"{output_prefix}_nemenyi_pvalues.csv"
            )

        # --------------------------------------------------------
        # 6. Critical-difference diagram
        # --------------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        # thicker lines + larger text
        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)

        for text in ax.texts:
            text.set_fontsize(17)

        # ax.set_title(
        #     f"Overall {metric_name} Critical Difference Diagram\n"
        #     f"Friedman p = {p_value:.2e}",
        #     fontsize=18
        # )

        plt.tight_layout()

        if output_prefix is not None:
            plt.savefig(
                f"{output_prefix}_critical_difference.png",
                dpi=300,
                bbox_inches="tight"
            )

        plt.show()
        plt.close()

    else:
        p_matrix = None
        print(
            "\nNo significant overall difference detected."
        )

    # ------------------------------------------------------------
    # 7. Save average ranks
    # ------------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value
    rank_df["n_datasets"] = len(complete)

    if output_prefix is not None:
        rank_df.to_csv(
            f"{output_prefix}_average_ranks.csv",
            index=False
        )

    return {
        "pivot": complete,
        "ranks": ranks,
        "average_ranks": avg_ranks,
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "nemenyi": p_matrix
    }

rmse_results = overall_significance_analysis(
    csv_path=Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/rmse_summary_all_models.csv"),
    metric_col="rmse",
    metric_name="RMSE",
    higher_is_better=False,
    output_prefix="rmse_overall"
)

qss_results = overall_significance_analysis(
    csv_path=Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}/qss_summary_all_models.csv"),
    metric_col="qss_0.1",
    metric_name="QSS",
    higher_is_better=True,
    output_prefix="qss_overall"
)

# Horizon-specific analysis
## CRPS and MASE

In [ ]:
from pathlib import Path
import pandas as pd
import re


# ============================================================
# CONFIG
# ============================================================

RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}")

ASSET_FOLDERS = {
    "G7": "FX",
    "EM": "EM FX",
    "MSCI": "MSCI",
    "Rates": "Rates",
    "REIT": "REIT",
    "Equities": "Equities",
    "ETF": "ETF",
    "Crypto": "Crypto",
}

MODEL_NAMES = {
    "arima_standardized": "ARIMA",
    "lightgbm_standardized_train_test": "LightGBM",
    "lstm_standardized_train_test": "LSTM",
    "sundial_standardized_train_test": "Sundial",
    "lag-llama_standardized_train_test": "Lag-Llama",
    "chronos_standardized_train_test": "Chronos-T5", 
    "toto_standardized_train_test": "Toto", 
    "timesfm_standardized_train_test": "TimesFM", 
    "moirai_standardized_train_test": "Moirai",
    "moirai_moe_standardized_train_test": "Moirai-MoE",
    "garch_standardized_train_test": "GARCH(1,1)",
}


ANCHOR = "_no_preprocess_lookback-250_horizon-20_slide-1_"


# ============================================================
# HELPERS
# ============================================================

def extract_dataset_name(filename):
    """
    Example:
    test_USD_EUR Historical Data_no_preprocess_lookback-250_horizon-20_slide-1_...
    
    -> USD_EUR Historical Data
    """
    stem = Path(filename).stem

    if not stem.startswith("test_"):
        raise ValueError(f"Unexpected filename: {filename}")

    before_anchor = stem.split(ANCHOR, 1)[0]

    return before_anchor[len("test_"):]


def read_metric_by_horizon(file_path, metric):
    """
    Reads a *_mase_by_horizon.csv or *_crps_by_horizon.csv
    and returns:

        horizon | value

    Tries to handle slightly different column names automatically.
    """
    df = pd.read_csv(file_path)

    # Remove accidental saved dataframe-index columns
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]

    # --------------------------------------------------------
    # Find horizon column
    # --------------------------------------------------------
    horizon_candidates = [
        c for c in df.columns
        if str(c).lower() in {
            "horizon",
            "h",
            "forecast_horizon",
            "forecast_step",
            "step"
        }
    ]

    if horizon_candidates:
        horizon_col = horizon_candidates[0]
        horizons = df[horizon_col].to_numpy()

    else:
        # If there is no explicit horizon column, assume rows
        # correspond to h = 0, ..., 19
        horizons = range(len(df))

    # --------------------------------------------------------
    # Find metric column
    # --------------------------------------------------------
    metric_lower = metric.lower()

    metric_candidates = [
        c for c in df.columns
        if metric_lower in str(c).lower()
        and str(c).lower() not in {
            "horizon",
            "forecast_horizon"
        }
    ]

    if len(metric_candidates) == 1:
        value_col = metric_candidates[0]

    elif len(metric_candidates) > 1:
        raise ValueError(
            f"Multiple possible {metric} columns in {file_path}:\n"
            f"{metric_candidates}"
        )

    else:
        # Fallback: find numeric columns that aren't horizon columns
        numeric_cols = df.select_dtypes(include="number").columns.tolist()

        if horizon_candidates:
            numeric_cols = [
                c for c in numeric_cols
                if c != horizon_candidates[0]
            ]

        if len(numeric_cols) != 1:
            raise ValueError(
                f"Could not uniquely identify metric column in:\n"
                f"{file_path}\n"
                f"Columns: {list(df.columns)}"
            )

        value_col = numeric_cols[0]

    out = pd.DataFrame({
        "horizon": list(horizons),
        metric: df[value_col].to_numpy()
    })

    return out


def dataset_match_key(name):
    """
    Creates a temporary key ONLY for matching dataset names.

    Examples:
        'AUD_USD Historical Data' -> 'audusd'
        'aud_usd'                 -> 'audusd'
        'AUD-USD'                 -> 'audusd'
    """
    name = str(name).lower().strip()

    # Remove common suffix appearing in the normal filenames
    name = name.replace("historical data", "")

    # Ignore spaces, underscores, hyphens, punctuation, etc.
    name = re.sub(r"[^a-z0-9]", "", name)

    return name

def build_canonical_dataset_lookup(metric):

    lookup = {}

    suffix = f"_{metric}_by_horizon.csv"

    for model_dir, model in MODEL_NAMES.items():

        # Don't use GARCH filenames to establish canonical names
        if model.lower() == "garch":
            continue

        for asset_folder, asset_class in ASSET_FOLDERS.items():

            folder = RESULTS_DIR / model_dir / asset_folder

            if not folder.exists():
                continue

            for file_path in folder.glob(f"*{suffix}"):

                try:
                    dataset = extract_dataset_name(file_path.name)

                    key = dataset_match_key(dataset)

                    if key in lookup and lookup[key] != dataset:
                        print(
                            f"WARNING: duplicate matching key '{key}': "
                            f"'{lookup[key]}' vs '{dataset}'"
                        )
                    else:
                        lookup[key] = dataset

                except Exception:
                    pass

    return lookup

# ============================================================
# COLLECT ONE METRIC
# ============================================================

def collect_metric(metric):
    records = []

    canonical_lookup = build_canonical_dataset_lookup(metric)

    print(
        f"Found {len(canonical_lookup)} canonical dataset names "
        f"for {metric.upper()}"
    )

    suffix = f"_{metric}_by_horizon.csv"

   

    print(f"\nCollecting {metric.upper()}")
    print("=" * 70)

    for model_dir, model in MODEL_NAMES.items():

        #model = MODEL_NAMES[model_dir.name]

        # Only keep GARCH(1,1) files from the GARCH directory
        is_garch_dir = model == "GARCH(1,1)"

        for asset_folder, asset_class in ASSET_FOLDERS.items():

            folder = RESULTS_DIR / model_dir / asset_folder

            if not folder.exists():
                continue

            files = list(folder.glob(f"*{suffix}"))

            for file_path in files:

                # Skip GARCH variants other than (1,1)
                if is_garch_dir and "_garch_1_1_" not in file_path.name:
                    continue
            
                try:
                    if is_garch_dir:

                        stem = file_path.stem.lower()
                    
                        # Example:
                        # g7_aud_usd_garch_1_1_train_test_crps_by_horizon
                        #
                        # -> g7_aud_usd
                        raw_dataset = stem.split("_garch_1_1_")[0]
                    
                        # Remove asset prefix:
                        # g7_aud_usd -> aud_usd
                        prefix = asset_folder.lower() + "_"
                    
                        if raw_dataset.startswith(prefix):
                            raw_dataset = raw_dataset[len(prefix):]
                    
                        # Turn into comparison key
                        key = dataset_match_key(raw_dataset)
                    
                        # Retrieve canonical dataset name
                        if key not in canonical_lookup:
                            print(
                                f"WARNING: Could not match GARCH dataset "
                                f"'{raw_dataset}' ({file_path.name})"
                            )
                            continue
                    
                        dataset = canonical_lookup[key]
                    
                    else:
                        dataset = extract_dataset_name(file_path.name)
            
                    horizon_df = read_metric_by_horizon(
                        file_path,
                        metric
                    )
            
                    for _, row in horizon_df.iterrows():
                        records.append({
                            "dataset": dataset,
                            "asset_class": asset_class,
                            "asset_folder": asset_folder,
                            "model": model,
                            "horizon": int(row["horizon"] - 1),
                            metric: row[metric],
                        })
            
                except Exception as e:
                    print(f"\nERROR reading:")
                    print(file_path)
                    print(e)
                

    result = pd.DataFrame(records)

    if result.empty:
        raise RuntimeError(
            f"No {metric} horizon files were found."
        )

    result = result.sort_values(
        ["asset_class", "dataset", "model", "horizon"]
    ).reset_index(drop=True)

    return result


# ============================================================
# BUILD BOTH MASTER SUMMARIES
# ============================================================

mase = collect_metric("mase")
crps = collect_metric("crps")


# ============================================================
# SANITY CHECKS
# ============================================================

print("\nMASE summary")
print(mase.shape)
print("Datasets:", mase["dataset"].nunique())
print("Models:", mase["model"].nunique())
print("Horizons:", sorted(mase["horizon"].unique()))

print("\nCRPS summary")
print(crps.shape)
print("Datasets:", crps["dataset"].nunique())
print("Models:", crps["model"].nunique())
print("Horizons:", sorted(crps["horizon"].unique()))

garch_crps = crps[
    crps["model"] == "GARCH(1,1)"
]

print("\nGARCH CHECK")
print("-" * 50)
print("GARCH datasets:", garch_crps["dataset"].nunique())
print("GARCH rows:", len(garch_crps))
print("GARCH horizons:", sorted(garch_crps["horizon"].unique()))

all_datasets = set(
    crps.loc[
        crps["model"] != "GARCH(1,1)",
        "dataset"
    ].unique()
)

garch_datasets = set(
    crps.loc[
        crps["model"] == "GARCH(1,1)",
        "dataset"
    ].unique()
)

missing_garch = sorted(all_datasets - garch_datasets)

print(
    f"\nDatasets without GARCH: {len(missing_garch)}"
)

for ds in missing_garch:
    print("  ", ds)


# ============================================================
# SAVE MASTER FILES
# ============================================================

mase.to_csv(
    RESULTS_DIR / "all_models_mase_by_dataset_and_horizon.csv",
    index=False
)

crps.to_csv(
    RESULTS_DIR / "all_models_crps_by_dataset_and_horizon.csv",
    index=False
)


# ============================================================
# SAVE JUST h = 0, 4, 19
# ============================================================

SELECTED_HORIZONS = [0, 4, 19]

mase_selected = mase[
    mase["horizon"].isin(SELECTED_HORIZONS)
].copy()

crps_selected = crps[
    crps["horizon"].isin(SELECTED_HORIZONS)
].copy()

mase_selected.to_csv(
    RESULTS_DIR / "all_models_mase_h1_h5_h20.csv",
    index=False
)

crps_selected.to_csv(
    RESULTS_DIR / "all_models_crps_h1_h5_h20.csv",
    index=False
)

print("\nDone")

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIG
# ============================================================

RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}")

MASE_FILE = RESULTS_DIR / "all_models_mase_by_dataset_and_horizon.csv"
CRPS_FILE = RESULTS_DIR / "all_models_crps_by_dataset_and_horizon.csv"

HORIZONS = [0, 4, 19]
ALPHA = 0.05


# ============================================================
# ONE HORIZON / ONE METRIC
# ============================================================

def analyse_horizon(
    df,
    metric,
    horizon,
    output_dir
):
    print("\n" + "=" * 80)
    print(f"{metric.upper()} — HORIZON h={horizon}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Select this horizon
    # --------------------------------------------------------
    df_h = df[df["horizon"] == horizon].copy()

    print("Rows:", len(df_h))
    print("Datasets:", df_h["dataset"].nunique())
    print("Models:", df_h["model"].nunique())

    # --------------------------------------------------------
    # 2. Check for duplicate dataset/model entries
    # --------------------------------------------------------
    duplicates = df_h.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nWARNING: duplicate dataset/model rows found:")
        print(
            df_h.loc[
                duplicates,
                ["dataset", "model", metric]
            ].sort_values(["dataset", "model"])
        )

        raise ValueError(
            f"Duplicate dataset/model entries found for "
            f"{metric}, h={horizon}."
        )

    # --------------------------------------------------------
    # 3. Create complete-block matrix
    #
    # rows    = datasets
    # columns = models
    # values  = metric
    # --------------------------------------------------------
    pivot = df_h.pivot(
        index="dataset",
        columns="model",
        values=metric
    )

    print("\nOriginal pivot shape:", pivot.shape)

    # Show missing values before dropping anything
    missing_counts = pivot.isna().sum()

    if missing_counts.sum() > 0:
        print("\nMissing values by model:")
        print(
            missing_counts[
                missing_counts > 0
            ].sort_values(ascending=False)
        )

    # Friedman needs a complete block:
    # every included dataset must have every included model.
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "GARCH"})

    dropped = len(pivot) - len(complete)

    print(
        f"\nComplete datasets used: {len(complete)} / {len(pivot)}"
    )

    if dropped > 0:
        print(
            f"Dropped {dropped} datasets because at least "
            f"one model was missing."
        )

    if len(complete) < 2:
        raise ValueError(
            "Not enough complete datasets for significance testing."
        )

    # --------------------------------------------------------
    # 4. Rank models separately within each dataset
    #
    # Lower MASE / CRPS = better
    # therefore rank 1 = best
    # --------------------------------------------------------
    ranks = complete.rank(
        axis=1,
        ascending=True,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # --------------------------------------------------------
    # 5. Friedman omnibus test
    # --------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # --------------------------------------------------------
    # 6. Save average ranks
    # --------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["horizon"] = horizon
    rank_df["n_datasets"] = len(complete)
    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value

    rank_path = (
        output_dir /
        f"{metric}_h{horizon}_average_ranks.csv"
    )

    rank_df.to_csv(rank_path, index=False)

    # --------------------------------------------------------
    # 7. Nemenyi post-hoc
    # --------------------------------------------------------
    p_matrix = None

    if p_value < ALPHA:

        print(
            "\nSignificant overall difference detected. "
            "Running Nemenyi post-hoc test..."
        )

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-value matrix:")
        print(p_matrix)

        p_matrix.to_csv(
            output_dir /
            f"{metric}_h{horizon}_nemenyi_pvalues.csv"
        )

        # ----------------------------------------------------
        # 8. Critical-difference diagram
        # ----------------------------------------------------
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)

        # h=0 is forecast step 1, etc.
        forecast_step = horizon + 1

        # ax.set_title(
        #     f"{metric.upper()} — Forecast Step {forecast_step} "
        #     f"(h={horizon})\n"
        #     f"Friedman p = {p_value:.2e}"
        # )

        plt.tight_layout()

        figure_path = (
            output_dir /
            f"{metric}_h{horizon}_critical_difference.png"
        )

        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:
        print(
            "\nNo significant overall difference at "
            f"alpha={ALPHA}; Nemenyi results are not interpreted."
        )

    return {
        "metric": metric,
        "horizon": horizon,
        "n_datasets": len(complete),
        "n_models": complete.shape[1],
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "average_ranks": avg_ranks,
        "nemenyi": p_matrix,
    }


# ============================================================
# RUN ALL THREE HORIZONS
# ============================================================

mase = pd.read_csv(MASE_FILE)
crps = pd.read_csv(CRPS_FILE)

OUTPUT_DIR = RESULTS_DIR / "statistical_significance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_results = []


for metric, df in [
    ("mase", mase),
    ("crps", crps),
]:

    for h in HORIZONS:

        result = analyse_horizon(
            df=df,
            metric=metric,
            horizon=h,
            output_dir=OUTPUT_DIR
        )

        all_results.append({
            "metric": metric,
            "horizon": h,
            "forecast_step": h + 1,
            "n_datasets": result["n_datasets"],
            "n_models": result["n_models"],
            "friedman_statistic": result["friedman_statistic"],
            "friedman_pvalue": result["friedman_pvalue"],
        })


# ============================================================
# SAVE OVERALL FRIEDMAN SUMMARY
# ============================================================

friedman_summary = pd.DataFrame(all_results)

friedman_summary.to_csv(
    OUTPUT_DIR / "friedman_horizon_summary.csv",
    index=False
)

print("\n\nFINAL FRIEDMAN SUMMARY")
print("=" * 80)
print(friedman_summary)

## QSS and RMSE

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import friedmanchisquare
import scikit_posthocs as sp


# ============================================================
# CONFIG
# ============================================================

RESULTS_DIR = Path(f"BForTFin_results/standardised/output_{REVIEW_FREQUENCY}")

RMSE_FILE = RESULTS_DIR / "rmse_by_horizon_all_models.csv"
QSS_FILE = RESULTS_DIR / "qss_by_horizon_all_models.csv"

HORIZONS = [1, 5, 20]
ALPHA = 0.05


# ============================================================
# ONE HORIZON / ONE METRIC
# ============================================================

def analyse_horizon(
    df,
    metric,
    horizon,
    output_dir
):
    print("\n" + "=" * 80)
    print(f"{metric.upper()} — HORIZON h={horizon}")
    print("=" * 80)

    # --------------------------------------------------------
    # 1. Select this horizon
    # --------------------------------------------------------
    df_h = df[df["horizon"] == horizon].copy()

    print("Rows:", len(df_h))
    print("Datasets:", df_h["dataset"].nunique())
    print("Models:", df_h["model"].nunique())

    # --------------------------------------------------------
    # 2. Check for duplicate dataset/model entries
    # --------------------------------------------------------
    duplicates = df_h.duplicated(
        subset=["dataset", "model"],
        keep=False
    )

    if duplicates.any():
        print("\nWARNING: duplicate dataset/model rows found:")
        print(
            df_h.loc[
                duplicates,
                ["dataset", "model", metric]
            ].sort_values(["dataset", "model"])
        )

        raise ValueError(
            f"Duplicate dataset/model entries found for "
            f"{metric}, h={horizon-1}."
        )

    # --------------------------------------------------------
    # 3. Create complete-block matrix
    #
    # rows    = datasets
    # columns = models
    # values  = metric
    # --------------------------------------------------------
    pivot = df_h.pivot(
        index="dataset",
        columns="model",
        values=metric
    )

    print("\nOriginal pivot shape:", pivot.shape)

    # Show missing values before dropping anything
    missing_counts = pivot.isna().sum()

    if missing_counts.sum() > 0:
        print("\nMissing values by model:")
        print(
            missing_counts[
                missing_counts > 0
            ].sort_values(ascending=False)
        )

    # Friedman needs a complete block:
    # every included dataset must have every included model.
    complete = pivot.dropna(axis=0, how="any").rename(columns={"Chronos-T5": "Chronos" , "GARCH(1,1)": "GARCH"})

    dropped = len(pivot) - len(complete)

    print(
        f"\nComplete datasets used: {len(complete)} / {len(pivot)}"
    )

    if dropped > 0:
        print(
            f"Dropped {dropped} datasets because at least "
            f"one model was missing."
        )

    if len(complete) < 2:
        raise ValueError(
            "Not enough complete datasets for significance testing."
        )

    # --------------------------------------------------------
    # 4. Rank models separately within each dataset
    #
    # Lower RMSE = better
    # Higher QSS = better
    # --------------------------------------------------------
    if metric.lower() == "qss_0.1":
        ascending = False   # higher is better
    else:
        ascending = True    # lower is better
    
    ranks = complete.rank(
        axis=1,
        ascending=ascending,
        method="average"
    )

    avg_ranks = ranks.mean(axis=0).sort_values()

    print("\nAverage ranks:")
    print(avg_ranks)

    # --------------------------------------------------------
    # 5. Friedman omnibus test
    # --------------------------------------------------------
    stat, p_value = friedmanchisquare(
        *[
            complete[col].to_numpy()
            for col in complete.columns
        ]
    )

    print("\nFriedman test")
    print(f"Statistic = {stat:.6f}")
    print(f"p-value   = {p_value:.6e}")

    # --------------------------------------------------------
    # 6. Save average ranks
    # --------------------------------------------------------
    rank_df = (
        avg_ranks
        .rename("average_rank")
        .reset_index()
    )

    rank_df["horizon"] = horizon-1
    rank_df["n_datasets"] = len(complete)
    rank_df["friedman_statistic"] = stat
    rank_df["friedman_pvalue"] = p_value

    rank_path = (
        output_dir /
        f"{metric}_h{horizon-1}_average_ranks.csv"
    )

    rank_df.to_csv(rank_path, index=False)

    # --------------------------------------------------------
    # 7. Nemenyi post-hoc
    # --------------------------------------------------------
    p_matrix = None

    if p_value < ALPHA:

        print(
            "\nSignificant overall difference detected. "
            "Running Nemenyi post-hoc test..."
        )

        p_matrix = sp.posthoc_nemenyi_friedman(
            complete
        )

        print("\nNemenyi p-value matrix:")
        print(p_matrix)

        p_matrix.to_csv(
            output_dir /
            f"{metric}_h{horizon-1}_nemenyi_pvalues.csv"
        )

        # ----------------------------------------------------
        # 8. Critical-difference diagram
        # ----------------------------------------------------
        if metric == "qss_0.1":
            metric_name = metric.replace("_0.1", "")
        else:
            metric_name = metric
        
        fig, ax = plt.subplots(figsize=(12, 5))

        sp.critical_difference_diagram(
            ranks=avg_ranks,
            sig_matrix=p_matrix,
            ax=ax,
            text_h_margin=0.02
        )

        for line in ax.lines:
            line.set_linewidth(4)
            line.set_markersize(8)
        
        for text in ax.texts:
            text.set_fontsize(17)

        # h=0 is forecast step 1, etc.
        forecast_step = horizon 

        # ax.set_title(
        #     f"{metric_name.upper()} — Forecast Step {forecast_step} "
        #     f"(h={horizon-1})\n"
        #     f"Friedman p = {p_value:.2e}"
        # )

        plt.tight_layout()

        figure_path = (
            output_dir /
            f"{metric_name}_h{horizon-1}_critical_difference.png"
        )

        plt.savefig(
            figure_path,
            dpi=300,
            bbox_inches="tight"
        )

        plt.show()
        plt.close()

    else:
        print(
            "\nNo significant overall difference at "
            f"alpha={ALPHA}; Nemenyi results are not interpreted."
        )

    return {
        "metric": metric_name,
        "horizon": horizon-1,
        "n_datasets": len(complete),
        "n_models": complete.shape[1],
        "friedman_statistic": stat,
        "friedman_pvalue": p_value,
        "average_ranks": avg_ranks,
        "nemenyi": p_matrix,
    }


# ============================================================
# RUN ALL THREE HORIZONS
# ============================================================

rmse = pd.read_csv(RMSE_FILE)
qss = pd.read_csv(QSS_FILE)

OUTPUT_DIR = RESULTS_DIR / "statistical_significance"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

all_results = []


for metric, df in [
    ("rmse", rmse),
    ("qss_0.1", qss),
]:

    for h in HORIZONS:

        result = analyse_horizon(
            df=df,
            metric=metric,
            horizon=h,
            output_dir=OUTPUT_DIR
        )

        all_results.append({
            "metric": metric.replace("_0.1",""),
            "horizon": h-1,
            "forecast_step": h,
            "n_datasets": result["n_datasets"],
            "n_models": result["n_models"],
            "friedman_statistic": result["friedman_statistic"],
            "friedman_pvalue": result["friedman_pvalue"],
        })


# ============================================================
# SAVE OVERALL FRIEDMAN SUMMARY
# ============================================================

friedman_summary = pd.DataFrame(all_results)

friedman_summary.to_csv(
    OUTPUT_DIR / "friedman_qss_horizon_summary.csv",
    index=False
)

print("\n\nFINAL FRIEDMAN SUMMARY")
print("=" * 80)
print(friedman_summary)